# Data Preprocessing


## Libraries


In [8]:
import json
import math
from pathlib import Path

import kagglehub
import librosa
from tqdm import tqdm
from dotenv import load_dotenv
import torch


## GPU


In [9]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


Device: mps


## Data


### Download Data


In [10]:
load_dotenv()
download_path = Path(
    kagglehub.dataset_download(
        "andradaolteanu/gtzan-dataset-music-genre-classification"
    )
)

print(f"download path: {download_path}")
print(f"data contents: {list(download_path.iterdir())}")


download path: /Users/pranavrajan/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1
data contents: [PosixPath('/Users/pranavrajan/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data')]


### Constants


In [11]:
SAMPLE_RATE = 22050
DURATION = 30  # measured in seconds
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

In [12]:
def save_mfcc(
    dataset_path, json_path, n_mfcc=13, n_fft=2048, hop_length=512, num_segments=5
):
    data = {
        "mapping": [],
        "mfcc": [],
        "labels": [],
    }

    num_samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    expected_num_mfcc_vectors_per_segment = math.ceil(
        num_samples_per_segment / hop_length
    )

    genre_dirs = sorted(p for p in dataset_path.iterdir() if p.is_dir())

    # Outer progress bar: tracks overall progress across all genres
    genre_pbar = tqdm(genre_dirs, desc="Total Genres", unit="genre")

    for i, genre_dir in enumerate(genre_pbar):
        semantic_label = genre_dir.name
        data["mapping"].append(semantic_label)

        wav_files = sorted(genre_dir.glob("*.wav"))

        # Inner progress bar: tracks files within the current genre
        # leave=False removes the inner bar once that genre finishes
        file_pbar = tqdm(
            wav_files,
            desc=f"{semantic_label:<10}",
            leave=False,
            unit="track",
        )

        for file_path in file_pbar:
            try:
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
            except Exception as e:
                # Use tqdm.write instead of print to prevent breaking the bar display
                tqdm.write(f"Skipping corrupted file {file_path.name}: {e}")
                continue

            for s in range(num_segments):
                start_sample = num_samples_per_segment * s
                end_sample = start_sample + num_samples_per_segment

                mfcc = librosa.feature.mfcc(
                    y=signal[start_sample:end_sample],
                    sr=sr,
                    n_fft=n_fft,
                    n_mfcc=n_mfcc,
                    hop_length=hop_length,
                ).T

                if len(mfcc) == expected_num_mfcc_vectors_per_segment:
                    data["mfcc"].append(mfcc.tolist())
                    data["labels"].append(i)

    json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)

    tqdm.write("\nData Processing Finished!")
    tqdm.write("=====================")


In [13]:
json_path = Path("../data/audio-deep-learning/mfcc_data.json")
dataset_path = download_path / "Data" / "genres_original"
save_mfcc(dataset_path, json_path, num_segments=10)

Total Genres:   0%|          | 0/10 [00:00<?, ?genre/s]

Total Genres:  50%|█████     | 5/10 [00:15<00:15,  3.08s/genre]/var/folders/73/y8z3_15n0c3fnwk5fkhj6ql40000gn/T/ipykernel_12293/982283500.py:37: UserWarning: PySoundFile failed. Trying audioread instead.
  signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
/Users/pranavrajan/Desktop/engineering-practice/.venv/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
                                                               
Total Genres:  50%|█████     | 5/10 [00:17<00:15,  3.08s/genre]

Skipping corrupted file jazz.00054.wav: 


Total Genres: 100%|██████████| 10/10 [00:31<00:00,  3.14s/genre]



Data Processing Finished!
